# California Housing Analysis

## PROBLEM STATEMENT:
The objective is to perform a comprehensive analysis of the California Housing dataset to prepare it for Machine Learning tasks.
This involves fetching data, exploring its structure, identifying and handling outliers using statistical methods (IQR, Z-Score),
and finally performing Feature Scaling (Standardization).

## STEPS TO SOLVE THE PROBLEM:
1.  **Data Loading**: Fetch the dataset from Scikit-Learn's repository.
2.  **Exploratory Data Analysis (EDA)**: Understand the data shape, types, and statistics.
3.  **Outlier Detection**:
    *   Identify extreme values using Percentiles.
    *   Detect outliers using the Interquartile Range (IQR) method.
    *   Detect logical inconsistencies (e.g., Bedrooms > Rooms).
    *   check for Z-score deviations.
4.  **Data Cleaning**: Remove the identified outliers to improve model quality.
5.  **Feature Scaling**: Standardize the Cleaned data so all features contribute equally.

## EXPECTED OUTPUT:
*   A summary of the dataset (shape, head, describe).
*   Lists of identified outliers from different methods.
*   Boxplots visualizing the spread and outliers.
*   Final shape of the dataset after cleaning.
*   A sample of the scaled data (Mean ~ 0, Std ~ 1).

### Step 1: Imports

### Import Libraries
We import the necessary libraries: `pandas` for data manipulation, `numpy` for numerical operations, `matplotlib` for visualization, and `sklearn` for the dataset and preprocessing tools.

In [11]:
# =============================================================================
# IMPORT STATEMENTS
# =============================================================================

# 2.1 Definition: Import the 'fetch_california_housing' function.
# 2.2 Why: This function provides a direct way to download the standard dataset.
# 2.7 Output: A function object available for use.
from sklearn.datasets import fetch_california_housing

# 2.1 Definition: Import Pandas library.
# 2.2 Why: For data manipulation using DataFrames (tables).
# 2.7 Output: Module 'pd'.
import pandas as pd

# 2.1 Definition: Import NumPy library.
# 2.2 Why: For numerical operations (absolute values, math).
# 2.7 Output: Module 'np'.
import numpy as np

# 2.1 Definition: Import SSL library.
# 2.2 Why: To handle Secure Sockets Layer (HTTPS) verification.
# 2.7 Output: Module 'ssl'.
import ssl

# 2.1 Definition: Import Pyplot from Matplotlib.
# 2.2 Why: To generate visualizations (Boxplots).
# 2.7 Output: Module 'plt'.
import matplotlib.pyplot as plt

# 2.1 Definition: Import StandardScaler.
# 2.2 Why: To transform data to have Mean=0 and Variance=1.
# 2.7 Output: A class for scaling.
from sklearn.preprocessing import StandardScaler

### Step 2: Helper Functions
We define the logic for detecting and removing outliers here.

### Helper Function: Detect Outliers (IQR)
We define a helper function `detect_outliers_iqr` that uses the Interquartile Range (IQR) method to identify outliers. Any data point falling below Q1 - 1.5*IQR or above Q3 + 1.5*IQR is flagged.

In [12]:
def detect_outliers_iqr(df, column):
    """
    3.1 What: Detects rows that are outliers based on IQR.
    3.2 Why: To identify data points that are statistically far from the median.
    Sample Example: Data [1 .. 100] -> 100 is outlier.
    """
    
    # 2.1 Definition: Calculate 25th Percentile (Q1).
    Q1 = df[column].quantile(0.25)

    # 2.1 Definition: Calculate 75th Percentile (Q3).
    Q3 = df[column].quantile(0.75)

    # 2.1 Definition: Calculate Interquartile Range (IQR).
    IQR = Q3 - Q1

    # 2.1 Definition: Calculate Fences.
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # 2.1 Definition: Return outliers.
    return df[(df[column] < lower_bound) | (df[column] > upper_bound)]

### Helper Function: Remove Outliers (IQR)
We define `remove_outliers_iqr`, a function that filters the DataFrame to keep only the rows that fall within the valid IQR bounds, effectively removing the outliers.

In [13]:
def remove_outliers_iqr(df, column):
    """
    3.1 What: Filters out rows that are outliers.
    3.2 Why: To clean the dataset.
    """
    # (Logic repeats calculating bounds)
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # 2.1 Definition: Filter for 'Good' data (Inliers).
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

### Helper Function: Detect Outliers (Z-Score)
We define `detect_outliers_zscore` to identify outliers based on their deviation from the mean. Data points with a Z-score (absolute value) greater than a threshold (default 3) are considered outliers.

In [14]:
def detect_outliers_zscore(df, column, threshold=3):
    """
    3.1 What: Detects outliers using Z-Score (Standard Deviation).
    3.2 Why: Good for Normally Distributed data.
    """
    # 2.1 Definition: Calculate Z-Score formula: (Value - Mean) / StdDev.
    z_scores = (df[column] - df[column].mean()) / df[column].std()
    
    # 2.1 Definition: Filter by absolute threshold.
    return df[np.abs(z_scores) > threshold]

### Step 3: Data Loading & Preparation

### Load Dataset
We fetch the California Housing dataset from the Scikit-Learn repository. We also handle SSL context to ensure the download proceeds without errors.

In [15]:
# 2.1 Definition: Set SSL Context to Unverified (Fixes download errors).
ssl._create_default_https_context = ssl._create_unverified_context

# 2.1 Definition: Fetch Dataset from Sklearn.
california = fetch_california_housing(as_frame=True)

# 2.1 Definition: Separate & Merge Data.
# We combine X (Data) and y (Target) into one DataFrame for analysis.
X = california.data
y = california.target

df = X.copy()
df['MedHouseValue'] = y

### Step 4: Exploratory Data Analysis (EDA)
We start by inspecting the raw data.

### Check Dimensions
We print the shape of the DataFrame to understand the size of our dataset (number of rows and columns).

In [16]:
# 2.1 Definition: Print Shape (Rows, Cols).
print(f"Dataset shape: {df.shape}")

Dataset shape: (20640, 9)


### Preview Data
We use `head()` to display the first 5 rows, giving us a visual overview of the data structure and feature values.

In [17]:
# 2.1 Definition: Print Head (First 5 rows).
print("First five rows of the dataset:")
display(df.head())

First five rows of the dataset:


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseValue
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


### Statistical Summary
We use `describe()` to generate a statistical summary for each numerical column, including count, mean, standard deviation, and quartiles.

In [18]:
# 2.1 Definition: Describe (Statistical Summary).
# Shows Count, Mean, Std, Min, Max for every column.
print("\nSummary statistics of the dataset:")
display(df.describe())


Summary statistics of the dataset:


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseValue
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


### Data Information
We use `info()` to verify data types and check for non-null values, which is crucial for identifying missing data.

In [21]:
# 2.1 Definition: Info (Data Types & Non-Nulls).
print("\nDataFrame info:")
print(df.info())


DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MedInc         20640 non-null  float64
 1   HouseAge       20640 non-null  float64
 2   AveRooms       20640 non-null  float64
 3   AveBedrms      20640 non-null  float64
 4   Population     20640 non-null  float64
 5   AveOccup       20640 non-null  float64
 6   Latitude       20640 non-null  float64
 7   Longitude      20640 non-null  float64
 8   MedHouseValue  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB
None


### Step 5: Data Quality Checks (Nulls & Anomalies)

### Check for Missing Values
We calculate the sum of missing (null) values for each column to decide if imputation or removal is needed.

In [20]:
# 2.1 Definition: Check Nulls.
print("\nMissing values in the dataset:")
print(df.isnull().sum())


Missing values in the dataset:
MedInc           0
HouseAge         0
AveRooms         0
AveBedrms        0
Population       0
AveOccup         0
Latitude         0
Longitude        0
MedHouseValue    0
dtype: int64


### Global Outlier Check
We perform a preliminary check for logical outliers, such as negative house values or values exceeding the maximum cap.

In [22]:
# 2.1 Definition: Check Logical Extreme Outliers.
# Houses < 0 dollars or > 5 (500k cap) might be issues.
outliers = df[(df['MedHouseValue'] < 0) | (df['MedHouseValue'] > 5)]
print("\nPotential outliers in 'MedHouseValue' (Global Check):")
print(outliers)


Potential outliers in 'MedHouseValue' (Global Check):
        MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
89      1.2434      52.0  2.929412   0.917647       396.0  4.658824     37.80   
459     1.1696      52.0  2.436000   0.944000      1349.0  5.396000     37.87   
493     7.8521      52.0  7.794393   1.051402       517.0  2.415888     37.86   
494     9.3959      52.0  7.512097   0.955645      1366.0  2.754032     37.85   
509     7.8772      52.0  8.282548   1.049861       947.0  2.623269     37.83   
...        ...       ...       ...        ...         ...       ...       ...   
20422   5.1457      35.0  6.958333   1.217593       576.0  2.666667     34.14   
20426  10.0472      11.0  9.890756   1.159664       415.0  3.487395     34.18   
20427   8.6499       4.0  7.236059   1.032528      5495.0  2.553439     34.19   
20436  12.5420      10.0  9.873315   1.102426      1179.0  3.177898     34.21   
20443   3.3438      50.0  5.342857   0.942857       13

### Step 6: Method 1 - Percentiles

### Method 1: Percentile Analysis
We calculate specific percentiles (1%, 25%, 50%, 75%, 99%) to analyze the distribution and detect potential capping of values.

In [23]:
print("\n=== METHOD 1: Extreme Values (Percentiles) ===")

# 2.1 Definition: Calculate Quantiles at 1%, 25%, 50%, 75%, 99%.
# 2.2 Why: To see if the data is capped or has long tails.
print(df['MedHouseValue'].quantile([0.01, 0.25, 0.5, 0.75, 0.99]))


=== METHOD 1: Extreme Values (Percentiles) ===
0.01    0.50000
0.25    1.19600
0.50    1.79700
0.75    2.64725
0.99    5.00001
Name: MedHouseValue, dtype: float64


### Step 7: Method 2 - IQR Method

### Method 2: IQR Analysis
We apply the IQR method specifically to the 'MedHouseValue' column to detect and count outliers based on the statistical range.

In [24]:
print("\n=== METHOD 2: IQR Method ===")

Q1 = df['MedHouseValue'].quantile(0.25)
Q3 = df['MedHouseValue'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 2.1 Definition: Find Price Outliers using IQR logic.
iqr_outliers = df[(df['MedHouseValue'] < lower_bound) | (df['MedHouseValue'] > upper_bound)]
print(f"Outliers detected: {len(iqr_outliers)}")
print(iqr_outliers.head())


=== METHOD 2: IQR Method ===
Outliers detected: 1071
     MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
89   1.2434      52.0  2.929412   0.917647       396.0  4.658824     37.80   
140  6.3624      30.0  5.615385   0.730769       126.0  2.423077     37.81   
459  1.1696      52.0  2.436000   0.944000      1349.0  5.396000     37.87   
489  3.0417      48.0  4.690632   1.126362      1656.0  3.607843     37.86   
493  7.8521      52.0  7.794393   1.051402       517.0  2.415888     37.86   

     Longitude  MedHouseValue  
89     -122.27        5.00001  
140    -122.18        4.83300  
459    -122.25        5.00001  
489    -122.25        4.89600  
493    -122.24        5.00001  


### Step 8: Method 3 - Zero/Negative check

### Method 3: Zero/Negative Value Check
We check for logically impossible values, such as negative age or zero income, which indicate data errors.

In [ ]:
print("\n=== METHOD 3: Zero or Negative Values ===")

# 2.1 Definition: Check impossible physical values (Age < 0, etc).
zero_negative = df[(df['MedInc'] <= 0) | (df['HouseAge'] < 0) | (df['Population'] <= 0)]
print(f"Zero/Negative values found: {len(zero_negative)}")
print(zero_negative)

### Step 9: Method 4 - Capped Values Check

### Method 4: Capped Value Check
We investigate if there is a 'ceiling' effect in the target variable, where many values are capped at the maximum (e.g., 5.0).

In [ ]:
print("\n=== METHOD 4: Capped/Repeated Maximum Values ===")

# 2.1 Definition: Check if many rows sit exactly at the Max Value.
capped = df[df['MedHouseValue'] == df['MedHouseValue'].max()]
print(f"Rows with maximum value (5.0): {len(capped)}")
print(capped.head())

### Step 10: Method 5 - Logical Inconsistencies

### Method 5: Logical Inconsistency Check
We verify logical relationships between features, such as ensuring the number of bedrooms does not exceed the total number of rooms.

In [ ]:
print("\n=== METHOD 5: Logical Inconsistencies ===")

# 2.1 Definition: A house cannot have more Bedrooms than total Rooms.
logical_issues = df[df['AveBedrms'] > df['AveRooms']]
print(f"Records where bedrooms > rooms: {len(logical_issues)}")
print(logical_issues)

### Step 11: Vizualization

### Visualization: Boxplots
We look at boxplots for 'AveRooms' and 'Population' to visually confirm the presence and spread of outliers.

In [ ]:
# 2.1 Definition: Boxplot.
# 2.2 Why: To visually confirm outliers in AveRooms and Population.
plt.figure()
df.boxplot(column=['AveRooms', 'Population'])
plt.title("Boxplots for Outlier Detection")
plt.show()

### Step 12: Cleaning Pipeline

### Cleaning Pipeline: detection
Before removing, we specifically detect and count outliers in the 'AveRooms' and 'Population' columns using the IQR method.

In [ ]:
# 2.1 Definition: Detect outliers in specific columns for logging.
ave_rooms_outliers = detect_outliers_iqr(df, 'AveRooms')
population_outliers = detect_outliers_iqr(df, 'Population')

print("Number of AveRooms outliers (IQR):", len(ave_rooms_outliers))
print("Number of Population outliers (IQR):", len(population_outliers))

# Optional Z-Score check for comparison
detect_outliers_zscore(df, 'Population')

### Cleaning Pipeline: Removal
We sequentially remove the identified outliers from 'AveRooms' and then 'Population' to create a clean dataset for analysis.

In [ ]:
# 2.1 Definition: Apply removal.
# 2.2 Why: To create the 'Gold Standard' dataset for training.
df_cleaned = remove_outliers_iqr(df, 'AveRooms')

# 2.1 Definition: Sequential cleaning (Chain the operations).
df_cleaned = remove_outliers_iqr(df_cleaned, 'Population')

print("Shape before outlier removal:", df.shape)
print("Shape after outlier removal:", df_cleaned.shape)

### Step 13: Feature Scaling
Standardizing the features to have Mean=0 and Std=1

### Feature Scaling: Preparation
We separate the features (X) from the target variable (y) because we typically only scale the features, not the target.

In [ ]:
# 2.1 Definition: Drop Target from Scaling set.
features = df_cleaned.drop(columns=['MedHouseValue'])
target = df_cleaned['MedHouseValue']

# 2.1 Definition: Initialize Scaler.
scaler = StandardScaler()

# 2.1 Definition: Fit and Transform.
scaled_features = scaler.fit_transform(features)

# 2.1 Definition: Reconstruct DataFrame.
df_scaled = pd.DataFrame(scaled_features, columns=features.columns)

# 2.1 Definition: Add Target back.
df_scaled['MedHouseValue'] = target.values

print("Scaled feature sample:")
display(df_scaled.head())